# Sprint 0 - Data Acquisition & Exploration

**Project:** Smart Crop Disease Detection and Assistant (see `AGENT.md` / `final_brief_and_plan.md`)

**Goal of this notebook (Sprint 0 "done when"):** download both datasets, organize them into
`train/val/test` folders by class, and load a labeled batch of images.

| Dataset | Source | Images | Classes |
|---|---|---|---|
| PlantVillage | GitHub `spMohanty/PlantVillage-Dataset` (raw/color) | ~54,300 | 38 |
| PlantDoc | GitHub `pratikkayal/PlantDoc-Dataset` | ~2,600 | 28 |

**How to use:** `Runtime -> Run all`. Data is saved to your Google Drive so it survives
session disconnects (Colab sessions reset every few hours).

**Pipeline:**
1. Mount Drive + clone this repo
2. Install dependencies
3. `scripts/download_datasets.py` -> `data/plantvillage/raw`, `data/plantdoc/raw`
4. `scripts/organize_datasets.py` -> `{train,val,test}/<class>` + `data/class_map.json`
5. Load a batch of images with `torchvision.ImageFolder` + `DataLoader` and inspect it

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip() or gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Google Drive and clone the repo

- **Drive** holds `folium/data` so the ~1 GB of images persists across Colab sessions.
- **Repo** is cloned fresh (or reused) so the latest scripts under `scripts/` are available.
- `REPO_URL` must be your own fork if you changed the remote.

In [ ]:
from google.colab import drive

# Mount Google Drive so the data persists across sessions.
# A popup asks you to authorize - click through and allow access.
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"   # change if you forked
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")

if not (REPO_DIR / "scripts").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    print(f"repo already cloned at {REPO_DIR}")

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR)

## Step 2 - Install dependencies

Training deps (`torch`, `torchvision`, `albumentations`) are installed now so we
don't have to wait again in Sprint 1.

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless

## Step 3 - Download the datasets

Runs `scripts/download_datasets.py`:
- **PlantVillage**: sparse checkout of `raw/color` from GitHub (`spMohanty/PlantVillage-Dataset`)
  - downloads only the ~1 GB of color images (no 2 GB data.zip, no HF loading script)
- **PlantDoc**: `git clone` of `pratikkayal/PlantDoc-Dataset`

The ~1 GB PlantVillage checkout runs in `WORK_DIR` (session scratch, `/content/folium_cache`)
so Google Drive isn't cluttered; the final color images land under `DATA_DIR`.
If a dataset already exists it is skipped (use `--force` to re-download).

In [ ]:
import subprocess
import sys

WORK_DIR = Path("/content/folium_cache")  # session-only scratch (PlantVillage ~1 GB checkout)
WORK_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(REPO_DIR / "scripts" / "download_datasets.py"),
    "--data-dir", str(DATA_DIR),
    "--work-dir", str(WORK_DIR),
    "--dataset", "all",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "download_datasets.py failed" 

## Step 4 - Organize into train/val/test

Runs `scripts/organize_datasets.py`:
- **PlantVillage**: stratified 80/10/10 split (seed 42)
- **PlantDoc**: keeps its shipped test set, carves 10% of training images as validation
- Writes `data/class_map.json` (PlantDoc -> PlantVillage class mapping)

In [ ]:
result = subprocess.run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--data-dir", str(DATA_DIR),
], cwd=str(REPO_DIR))
assert result.returncode == 0, "organize_datasets.py failed" 

## Step 5 - Verify: load a labeled batch

The "done" check for Sprint 0: load images + labels with `torchvision.ImageFolder`
and pull one batch from a `DataLoader`.

In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])
pv_train = datasets.ImageFolder(DATA_DIR / "plantvillage" / "train", transform=transform)
loader = DataLoader(pv_train, batch_size=32, shuffle=True, num_workers=2)

images, labels = next(iter(loader))
print("One batch: images", tuple(images.shape), "labels", tuple(labels.shape))
print("dtype/range: images", images.dtype, round(float(images.min()), 2), "-", round(float(images.max()), 2))
print("num train classes:", len(pv_train.classes))
print("num train images:", len(pv_train))
assert len(pv_train) > 0 and len(pv_train.classes) == 38
print("SPRINT 0 DONE: can load a labeled batch of images")

### Sample images from the batch (resized to 256x256)

In [ ]:
import matplotlib.pyplot as plt
import torchvision.utils as vutils

grid = vutils.make_grid(images[:16], nrow=4, normalize=True).permute(1, 2, 0).numpy()
plt.figure(figsize=(12, 12))
plt.imshow(grid)
plt.axis("off")
plt.title("First 16 images (resized 256x256)")
plt.show()

for i in range(8):
    print(f"  {i}: {pv_train.classes[labels[i].item()]}")

### Train class distribution (PlantVillage)

In [ ]:
import collections

counts = collections.Counter(pv_train.targets)
ordered = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
names = [pv_train.classes[i] for i, _ in ordered]
values = [v for _, v in ordered]

plt.figure(figsize=(16, 5))
plt.bar(names, values)
plt.xticks(rotation=90, fontsize=8)
plt.ylabel("images")
plt.title("PlantVillage train class distribution (n = %d)" % len(pv_train))
plt.show()

## Where things live

```
<DATA_DIR>/
  plantvillage/{train,val,test}/<class>/*.jpg    38 classes, 80/10/10
  plantdoc/{train,val,test}/<class>/*.jpg        28 classes, shipped test + 90/10
  class_map.json                                  PlantDoc -> PlantVillage alignment
```

**Notes**
- `WORK_DIR` (`/content/folium_cache`) is session-only scratch — it is wiped when the
  Colab session resets, which is expected. The organized data on Drive persists.
- Run `organize` again after any fresh download - it is idempotent (overwrites).
- The PlantDoc clone loses 6 files on case-insensitive filesystems - we are on Colab
  (Linux), so all 2,578 images are kept. The script warns if run on macOS/Windows.